# Reproduce OPT-031 from the F_BOTH checkpoint
The fixed pipeline converts each official H-alpha JPEG to grayscale RGB, predicts on identity/H/V views, restores coordinates, fuses at confidence 0.90 and pixel cutoff 0.40, resolves overlap and serializes COCO RLE. See the README for the pinned CUDA environment and the report for the training split and loss.

In [ ]:
import os, json, subprocess, sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
manifest = json.loads((ROOT / "evidence/manifest.json").read_text())
manifest

## Verify the archived prediction file
This section uses CPU only. The inventory and CSV are included.

In [ ]:
subprocess.run([sys.executable, "reproduce.py", "--validate-only",
                "evidence/submission-opt031-c090-p040.csv"], cwd=ROOT, check=True)

## Regenerate on the official test inputs
Set `MAGFILO_TEST_IMAGES` to the official unlabelled test image directory. This cell uses CUDA and loads the model-only checkpoint from Git LFS.

In [ ]:
test_images = Path(os.environ["MAGFILO_TEST_IMAGES"]).expanduser().resolve()
subprocess.run([sys.executable, "reproduce.py", "--test-images", str(test_images),
                "--output", "predictions/submission.csv"], cwd=ROOT, check=True)

## Development metrics
The files below record the historical evaluation on 148 development photos.

In [ ]:
json.loads((ROOT / "evidence/development-148-metrics.json").read_text())